In [6]:
import mlflow

mlflow.set_tracking_uri(
    "sqlite:///../mlflow.db"
)

mlflow.set_experiment(
    "Cricket Winner Prediction"
)

#Loading the datasets

import pandas as pd 
df = pd.read_csv("../data/processed/matches_features_v1.csv")

#Seprating the data from target

X = df.drop(["winner","date"], axis=1)
y = df["winner"]

#Spliting the data for training , testing and valdation

from sklearn.model_selection import train_test_split as tts 

split_idx = int(len(df) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

# print(X_train.shape)
# print(X_test.shape)

# print(df.iloc[:split_idx]["date"].min())
# print(df.iloc[:split_idx]["date"].max())

# print(df.iloc[split_idx:]["date"].min())
# print(df.iloc[split_idx:]["date"].max())


#Preprocessig the Data

from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

categorical_features = ["team1", "team2","venue","toss_decision","toss_winner"]

numerical_features = ["team1_matches_played",
                      "team1_win_rate",
                      "team1_form_5",

    "team2_matches_played",
    "team2_win_rate",
    "team2_form_5",

    "team1_h2h_wins",
    "team2_h2h_wins",

    "team1_venue_matches",
    "team1_venue_win_rate",

    "team2_venue_matches",
    "team2_venue_win_rate",

    "team1_won_toss",
    "team2_won_toss"
    ]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat", OneHotEncoder(handle_unknown="ignore"),categorical_features,
            
        ),
        (
            "num", StandardScaler(), numerical_features
        )
    ],
    remainder="passthrough"
)

#Model Selection and fitting

from sklearn.linear_model import LogisticRegression

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier",LogisticRegression(
        max_iter=1000,
        random_state=42
)
        )
    ]
)



#Model metrics validation

from sklearn.metrics import accuracy_score, recall_score, f1_score 
from sklearn.metrics import classification_report, confusion_matrix

# accuracy = accuracy_score(y_test, y_pred)
# print("accuracy is ", accuracy)
# print(classification_report(y_test, y_pred))
# con = confusion_matrix(y_test,y_pred)
# print(con)

#NOW THE FUN PART(mlflow)

import mlflow
import mlflow.sklearn


with mlflow.start_run(run_name = "LogisticRegression_v2"):

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted"
    )
    print("Accuracy",accuracy)
    # print(classification_report(y_test, y_pred))

    mlflow.log_param("model", "LogisticRegression_V2")
    mlflow.log_param("max_iter", 1000)
    #mlflow.log_params("C", 2)
    mlflow.log_param("Random State", 42)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_weighted", f1)

    mlflow.sklearn.log_model(
        model,
        artifact_path="model"
    )
  

2026/06/19 12:06:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy 0.6609360076408787


2026/06/19 12:06:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=15,
                random_state=42
            )
        )
    ]
)

with mlflow.start_run(run_name="RandomForest"):

    rf_model.fit(X_train, y_train)

    y_pred = rf_model.predict(X_test)

    accuracy_rf = accuracy_score(y_test, y_pred)

    f1_rf = f1_score(
        y_test,
        y_pred,
        average="weighted"
    )

    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 500)
    mlflow.log_param("max_depth", 15)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_weighted", f1)

    mlflow.sklearn.log_model(
        rf_model,
        name="model"
    )

    print(accuracy_rf)
    print(f1_rf)
    print("Random Forest Accuracy:", accuracy_rf)

    train_acc = accuracy_score(
        y_train,
        rf_model.predict(X_train)
)   
    print("Train Accuracy:", train_acc)  



2026/06/18 14:24:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


0.5109837631327603
0.4653866171140736
Random Forest Accuracy: 0.5109837631327603
Train Accuracy: 0.8187679083094556


In [ ]:
# f
# rom sklearn.preprocessing import LabelEncoder

# le = LabelEncoder()

# le.fit(y)

# y_train_enc = le.transform(y_train)
# y_test_enc = le.transform(y_test)
# xgb_model.fit(X_train, y_train_enc)

# y_pred_enc = xgb_model.predict(X_test)

# accuracy = accuracy_score(
#     y_test_enc,
#     y_pred_enc
# )

In [ ]:
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn
from sklearn.preprocessing import LabelEncoder
xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.1,
                random_state=42,
                eval_metric="mlogloss"
            )
        )
    ]
)

with mlflow.start_run(run_name="XGBoost"):
   

    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)


    seen_classes = set(y_train)

    mask = y_test.isin(seen_classes)

    X_test_filtered = X_test[mask]
    y_test_filtered = y_test[mask]
    y_test_enc = le.fit_transform(y_test_filtered)

    xgb_model.fit(X_train, y_train_enc)

    y_pred = xgb_model.predict(X_test_filtered)

    accuracy = accuracy_score(y_test_enc, y_pred)

    f1 = f1_score(
        y_test_enc,
        y_pred,
        average="weighted"
    )

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_weighted", f1)

    mlflow.sklearn.log_model(
        xgb_model,
        name="model"
    )

    print("Accuracy:", accuracy) 

2026/06/18 14:15:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.03861003861003861
